# 教学演示：酒店推荐 Agent 与工具调用

**本课三个重点**：
1. 用 `@tool` 定义 `recommend_hotel`，对接百度地图 POI 查酒店列表。
2. 用 `create_agent` 把「大模型 + 工具 + 系统提示词」组装成 Agent。
3. 用 `astream_events` 观察 **Agent 何时自动调工具**、工具返回什么、模型如何选一家酒店回复用户。

> 地图 HTTP 细节在 `travel_common.py`；本 notebook 只聚焦 Agent 工具调用链路。

## 0. 环境

```bash
pip install langchain langchain-openai langgraph python-dotenv httpx
```

在项目根目录 `.env` 中配置：

- `DASHSCOPE_API_KEY` — 百炼大模型
- `BAIDU_MAP_AK` — 百度地图 Place API（查酒店）

对应脚本：`hotel_recommendation_demo.py`

In [23]:
# Windows 下避免 print 中文乱码（可选）
import sys

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

## 1. 课堂示例：用户 Query 与系统提示词

可改成你自己的城市/偏好。工具**不做规则过滤**，返回 `hotels` 列表后由大模型挑选一家。

In [24]:
USER_QUERY = "我要去大同玩三天给我推荐酒店，需要安静/近景区"

SYSTEM_INSTRUCTION = """你是酒店推荐助手，只能通过工具 recommend_hotel 查酒店。
规则：
1. city 必填。
2. preferences 只填「地图能搜的关键词」：区名、景点、地标、酒店品牌（如 近景区、平城区、希尔顿）。
   不要填主观感受（安静、舒适、性价比）——这些留给你读 hotels 后再判断。
3. 工具返回 hotels 后，结合用户全部诉求，只推荐最合适的一家。
4. 非酒店问题，回复：我只能协助酒店推荐。"""

print("用户:", USER_QUERY)

用户: 我要去大同玩三天给我推荐酒店，需要安静/近景区


## 2. 依赖与地图 API（引用 travel_common）

In [25]:
import json
import os
import sys
from typing import Any, Dict, List, Optional

import httpx
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# notebook 在项目根目录运行时，加入 enterprise_bench 以便 import agents.*
_ROOT = os.path.abspath(os.getcwd())
_ENT = os.path.join(_ROOT, "enterprise_bench")
if _ENT not in sys.path:
    sys.path.insert(0, _ENT)

from travel_common import (
    ensure_project_dotenv_loaded,
    fetch_hotels_from_api,
    norm_text,
    require_non_empty,
)

load_dotenv()
ensure_project_dotenv_loaded()
print("BAIDU_MAP_AK:", "已配置" if os.getenv("BAIDU_MAP_AK") else "未配置")
print("DASHSCOPE_API_KEY:", "已配置" if os.getenv("DASHSCOPE_API_KEY") else "未配置")

BAIDU_MAP_AK: 已配置
DASHSCOPE_API_KEY: 已配置


## 3. 定义工具 `recommend_hotel`

**注意**：百度 POI 只认地点/品牌。把 `安静,近景区` 整段当搜索词会返回「黄石市」等地名。

- **进地图搜索**：`近景区`、`希尔顿` → `近景区 酒店`
- **不进地图搜索**：`安静`、`舒适` → 仅用 `酒店` 搜同城，由大模型结合 preferences 再选

In [28]:
import re

_DISTRICT_RE = re.compile(r"[\u4e00-\u9fff]{1,12}(?:区|县)")
_POI_HINT = re.compile(
    r"区|县|景区|景点|古城|公园|广场|山|湖|寺|庙|步行街|地铁|高铁|火车|机场|"
    r"希尔顿|万豪|洲际|如家|全季|亚朵|汉庭|民宿"
)
_SUBJECTIVE_HINT = re.compile(r"安静|吵闹|性价比|舒适|干净|卫生|便宜|奢华|亲子|早餐|贴心|便利|方便")


def poi_search_keyword(preferences: Optional[str]) -> str:
    pref = norm_text(preferences)
    if not pref:
        return "酒店"
    if _POI_HINT.search(pref) or _DISTRICT_RE.search(pref):
        return pref if "酒店" in pref else f"{pref} 酒店"
    if _SUBJECTIVE_HINT.search(pref) or "," in pref or "，" in pref:
        return "酒店"
    return pref if "酒店" in pref else f"{pref} 酒店"


def _valid_hotel_poi(h: Dict[str, Any]) -> bool:
    name = (h.get("name") or "").strip()
    if not name:
        return False
    if name.endswith("市") and not h.get("address") and not h.get("district"):
        return False
    return True


@tool
async def recommend_hotel(
    city: str,
    preferences: Optional[str] = None,
    budget_cny_per_night_max: Optional[int] = None,
) -> Dict[str, Any]:
    """查酒店列表。preferences 传区名/景点/品牌；安静等主观词由模型读 hotels 后判断。"""
    ok, err = require_non_empty(city, "city")
    if not ok:
        return {"error": err}

    pref = norm_text(preferences)
    keyword = poi_search_keyword(pref)

    hotels: List[Dict[str, Any]] = []
    source = "none"
    res = await fetch_hotels_from_api(city, limit=10, keyword=keyword)
    if not res.get("error"):
        source = res.get("data_source", "api")
        for h in res.get("hotels") or []:
            if isinstance(h, dict):
                item = {k: v for k, v in h.items() if k != "raw"}
                if _valid_hotel_poi(item):
                    hotels.append(item)

    return {
        "city": city,
        "search_query": keyword,
        "preferences": pref or None,
        "budget_cny_per_night_max": budget_cny_per_night_max,
        "hotels": hotels,
        "data_source": source,
        "note": "请结合 preferences 与 hotels 选一家。",
    }

## 4. 组装 Agent（模型 + 工具 + 系统提示词）

In [29]:
llm = ChatOpenAI(
    model=os.getenv("DASHSCOPE_CHAT_MODEL", "qwen-plus"),
    temperature=0,
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv(
        "DASHSCOPE_CHAT_BASE_URL",
        "https://dashscope.aliyuncs.com/compatible-mode/v1",
    ).rstrip("/"),
    http_client=httpx.Client(verify=False),
)

agent = create_agent(
    llm,
    tools=[recommend_hotel],
    system_prompt=SYSTEM_INSTRUCTION,
    checkpointer=MemorySaver(),
)
print("Agent 已创建，绑定工具:", [t.name for t in [recommend_hotel]])

Agent 已创建，绑定工具: ['recommend_hotel']


## 5. 对比：直接调用工具（不经过 Agent）

帮助理解：工具本身只负责「查列表」，不负责最终推荐话术。

In [15]:
r = await recommend_hotel.ainvoke({"city": "大同", "preferences": "近景区"})
print("搜索词:", r["search_query"])
print("候选数量:", len(r["hotels"]))
for i, h in enumerate(r["hotels"], 1):
    print(f"  {i}. {h.get('name')} | {h.get('district') or h.get('address')}")

搜索词: 近景区 酒店
候选数量: 5
  1. 浑源旨岭宜景酒店(北岳恒山景区真武庙店) | 浑源县
  2. 遇见·恒山 | 浑源县
  3. 瑞福民宿(大同古城景区店) | 平城区
  4. 浑源恒禄民宿 | 浑源县
  5. 魏都水世界 | 平城区


## 6. 正文：Agent 自动调用工具

观察 `on_tool_start` / `on_tool_end`：模型自己决定何时调 `recommend_hotel`、传什么参数。

In [30]:
def short_json(data: Any, max_len: int = 800) -> str:
    if hasattr(data, "content"):
        data = data.content
    if isinstance(data, str):
        try:
            data = json.loads(data)
        except json.JSONDecodeError:
            return data[:max_len]
    s = json.dumps(data, ensure_ascii=False, indent=2)
    return s if len(s) <= max_len else s[:max_len] + "…"


print(f"用户: {USER_QUERY}\n")
print("模型: ", end="", flush=True)

async for ev in agent.astream_events(
    {"messages": [("user", USER_QUERY)]},
    {"configurable": {"thread_id": "hotel_nb_demo"}},
    version="v2",
):
    kind = ev["event"]
    if kind == "on_chat_model_stream" and ev["data"]["chunk"].content:
        print(ev["data"]["chunk"].content, end="", flush=True)
    elif kind == "on_tool_start":
        print(f"\n\n>>> 工具调用: {ev.get('name')}")
        print(">>> 参数:", short_json(ev["data"].get("input")))
    elif kind == "on_tool_end":
        print(">>> 工具返回:", short_json(ev["data"].get("output")))
        print("\n模型继续: ", end="", flush=True)
    elif kind == "on_chain_end" and ev.get("name") == "LangGraph":
        break

print()

用户: 我要去大同玩三天给我推荐酒店，需要安静/近景区

模型: 

>>> 工具调用: recommend_hotel
>>> 参数: {
  "city": "大同",
  "preferences": "景区"
}
>>> 工具返回: {
  "city": "大同",
  "search_query": "景区 酒店",
  "preferences": "景区",
  "budget_cny_per_night_max": null,
  "hotels": [
    {
      "name": "浑源旨岭宜景酒店(北岳恒山景区真武庙店)",
      "district": "浑源县",
      "address": "恒山景区真武庙旁边",
      "tel": "15525220428",
      "location": "113.740723,39.668781",
      "rating": 4.8,
      "avg_price_cny": null,
      "type": "hotel"
    },
    {
      "name": "遇见·恒山",
      "district": "浑源县",
      "address": "大同市恒山北路国际绿洲·和园七号楼一单元801",
      "tel": "13935273488",
      "location": "113.70018,39.71848",
      "rating": 4.8,
      "avg_price_cny": null,
      "type": "hotel"
    },
    {
      "name": "瑞福民宿(大同古城景区店)",
      "district": "平城区",
      "address": "山西省大同市平城区古城街道大同古城云路街20号",
      "tel": "15303524888",
      "location": "113.312935,40.092551",
      "rating…

模型继续: 根据您的需求（大同游玩三天、安静、近景区），我为您推荐：

**瑞福民宿(大同古城景区店)**  
- 位于**平城区大同古城云路街20号*

## 7. 小结

| 环节 | 作用 |
|------|------|
| `recommend_hotel` 工具 | 调地图 API，返回 `hotels` 候选 |
| `create_agent` | 把 LLM 和工具绑在一起 |
| `astream_events` | 流式看模型输出 + 工具调用事件 |
| 大模型 | 读 `hotels`，结合用户话选一家并生成自然语言推荐 |
